In [5]:
from scipy.io import loadmat    # used to turn .mat files into numpy Arrays
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [6]:
def plot_ABP_PPG_signal(ABP, PPG, fs):

    t = np.arange(len(PPG)) / fs    # time axis, in seconds 
    
    plt.figure(figsize=(12,6))

    plt.subplot(2,1,1)
    plt.plot(t, ABP, color='blue')
    plt.title('ABP Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Normalized ABP ([0,1])')

    plt.subplot(2,1,2)
    plt.plot(t, PPG, color='red')
    plt.title('PPG Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Light Intensity [V]')

    plt.tight_layout()

    plt.show()

In [7]:
def unwrap_matlab_array(x):
    '''
    Recursively unwrap MATLAB object arrays to a plain Python number
    '''
    x = np.asarray(x)
    while x.dtype == object:
        x = x[0]
    if np.size(x) == 1:
        return x.item()
    return x


In [15]:
data_path = 'ABP_PPG'
data_file = sorted(os.listdir(data_path))
rows = []
print(len(data_file))
for i in range(1000):
    file_path = os.path.join(data_path, data_file[i])
    data = loadmat(file_path)
    for j in range(1):
        # print(f'accessing struct {j}:')
        print(f'Accessing datapoint {i}')
        sp = data['signal_processing'][0,j]
        segment = sp['signal']

        ABP = segment[0]

        beats = sp['struct_features']['beats_abp'][0,0]
        beats_masked = np.where(beats == 0, np.nan, beats)  # Replace 0 padding with NaN
        #DBP = np.nanmin(beats_masked, axis=1).mean()
        #SBP = np.nanmax(beats_masked, axis=1).mean()
        
        '''
        # mean_pulse_abp aligns all the beats first and then calculates the mean values for each point
        beats_mean = sp['struct_features']['mean_pulse_abp'][0]
        # print(beats_mean)
        beats_mean = np.asarray(beats_mean[0], dtype=float)
        beats_mean_masked = np.where(beats_mean == 0, np.nan, beats_mean)
        # print(beats_mean_masked)
        DBP_mean = np.nanmin(beats_mean_masked, axis=1)
        SBP_mean = np.nanmax(beats_mean_masked, axis=1)
        print(f'means:{DBP_mean, SBP_mean}')

        #mean(min(beat)) != min(mean(beat))
        #mean(max(beat)) != max(mean(beat))
        '''
        
        feat_ppg = sp['struct_stats_features']['ppg'][0,0]
        ppg_mean_struct = feat_ppg['mean'][0, 0]
        #print(ppg_mean_struct)
        mean_sbp = unwrap_matlab_array(ppg_mean_struct['mean_sbp'])
        mean_dbp = unwrap_matlab_array(ppg_mean_struct['mean_dbp'])
        #print(mean_sbp)
        #print(mean_dbp)

        ppg_mean_dict = {}
        ppg_std_dict = {}
        for stat_name, target_dict in [('mean', ppg_mean_dict),('std', ppg_std_dict)]:
            stat_struct = feat_ppg[stat_name][0,0]
            for field in stat_struct.dtype.names:
                target_dict[f'ppg_{stat_name}_{field}'] = unwrap_matlab_array(stat_struct[field])

        row = {
            'data_ID': data_file[i],
            'struct': j,
            'SBP': mean_sbp,
            'DBP': mean_dbp
        }
        row.update(ppg_mean_dict)
        rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv('ABP_PPG_features.csv', index=False)


        '''
        delineator_abp = sp['struct_delineator']['ABP'][0,0]
        print(delineator_abp.dtype.names)
        sysPeak_idx = delineator_abp['SysPeak'][0,0]
        foot_idx = delineator_abp['foot'][0,0]
        

        # numpy.ndaaray's of the indices
        sysPeak_idx = sysPeak_idx[0]
        foot_idx = foot_idx[0]

        SBP = ABP[sysPeak_idx]
        DBP = ABP[foot_idx]
        SBP_mean = SBP.mean()
        DBP_mean = DBP.mean()
        
        feat = sp['struct_stats_features']['ppg'][0,0]
        print(feat.dtype.names)
        feat_mean = feat['mean'][0,0]
        feat_std = feat['std'][0,0]
        print(feat_mean)
        '''
        
        

5857
Accessing datapoint 0
Accessing datapoint 1
Accessing datapoint 2
Accessing datapoint 3
Accessing datapoint 4
Accessing datapoint 5
Accessing datapoint 6
Accessing datapoint 7
Accessing datapoint 8
Accessing datapoint 9
Accessing datapoint 10
Accessing datapoint 11
Accessing datapoint 12
Accessing datapoint 13
Accessing datapoint 14
Accessing datapoint 15
Accessing datapoint 16
Accessing datapoint 17
Accessing datapoint 18
Accessing datapoint 19
Accessing datapoint 20
Accessing datapoint 21
Accessing datapoint 22
Accessing datapoint 23
Accessing datapoint 24
Accessing datapoint 25
Accessing datapoint 26
Accessing datapoint 27
Accessing datapoint 28
Accessing datapoint 29
Accessing datapoint 30
Accessing datapoint 31
Accessing datapoint 32
Accessing datapoint 33
Accessing datapoint 34
Accessing datapoint 35
Accessing datapoint 36
Accessing datapoint 37
Accessing datapoint 38
Accessing datapoint 39
Accessing datapoint 40
Accessing datapoint 41
Accessing datapoint 42
Accessing datapo